In [ ]:
#imports

import pandas as pd 
import numpy as np 
import scipy as sp
import os 
import matplotlib.pyplot as plt
import keras
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.optimizers import Adam
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


In [ ]:
# creating directories for files upload

if not os.path.exists('CORONA'):
       os.mkdir ('CORONA')

if not os.path.exists('CORONA/SPECTROS'):
       os.mkdir ('CORONA/SPECTROS')


In [ ]:
#uploading spectrograms' files from PC into Colab database

from google.colab import files 
uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.png'):
        with open(filename, 'wb') as f:
            f.write(uploaded[filename])
        !mv {filename} CORONA/SPECTROS/
        

In [ ]:
# training neural network 

spectros = 'CORONA/SPECTROS'

label_map = {'neg': 0, 'pos': 1}

# load the spectrograms into memory
X = []
y = []

for fino in os.listdir(spectros):
    if fino.endswith('.png'): # load the spectrograms' file
        spect_path = os.path.join(spectros, fino)
        spect = plt.imread(spect_path)

        # add the spectrogram to X
        X.append(spect)

        # extract the label from the filename
        label = (fino.split('-')[0])
        label = label_map[label]

        # add the label to Y
        y.append(label)

# convert Y to a numpy array
y = np.array(y)

# one-hot encode the labels
y = to_categorical(y)

# convert X and Y to numpy arrays
X = np.array(X)
y = np.array(y)


#defining procentage of train/test data

train_data, test_data, train_labels, test_labels = train_test_split(
            X, Y, test_size=0.2, shuffle=True)



num_classes = 2

# define the neural network architecture
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=X.shape[1:]))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.3))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.6))
model.add(Dense(num_classes, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
trenado = model.fit(X, y, batch_size=32, epochs=50)


In [ ]:
#ploting the results of the training


# loss plot
plt.plot(trenado.history['loss'], color = 'red')
plt.xlabel('epochs')
plt.legend(['loss'])
plt.show()

# accuracy plot 
plt.plot(trenado.history['accuracy'], color='green')
plt.xlabel('epochs')
plt.legend(['accuracy'])
plt.show()


In [ ]:
train_loss, train_accuracy = model.evaluate (train_data, train_labels)
test_loss, test_accuracy = model.evaluate (test_data, test_labels)

print("Train Loss: %.4f"%train_loss)
print("Train Accuracy: %.4f"%train_accuracy) 
print("Test Loss: %.4f"%test_loss)
print("Test Accuracy: %.4f"%test_accuracy)
